# 02 · SQL analysis

**Question:** do free downloaders become paying customers, and how long does it take?

Every query lives in `sql/` as its own file, so it can be read and run on its own. This notebook runs them in order against `data/work/music_business.db` and saves the results the site uses.

**Keeping the business private.** Every query returns a rate, a share, a median or an index. None returns a count of customers, orders, subscribers or tracks, or any revenue. Shares based on small groups are rounded (to the nearest 5% or whole percent) so that nobody can work backwards from a percentage to the number behind it.

In [1]:
import json
import sqlite3
from pathlib import Path

import pandas as pd

ROOT = Path("..")
SQL = ROOT / "sql"
con = sqlite3.connect(ROOT / "data" / "work" / "music_business.db")

def run(name):
    return pd.read_sql((SQL / name).read_text(), con)

def round_to_total(values, step):
    # Round shares to multiples of `step` while keeping the total at 100 (largest remainder method).
    units = [v / step for v in values]
    floors = [int(u) for u in units]
    left = round(100 / step) - sum(floors)
    order = sorted(range(len(units)), key=lambda i: units[i] - floors[i], reverse=True)
    for i in order[:left]:
        floors[i] += 1
    return [f * step for f in floors]

con.executescript((SQL / "00_views.sql").read_text())
print("Views:", [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type = 'view'")])

Views: ['paid_items', 'paid_orders', 'buyers', 'buyer_paths']


## Definitions

- **Paid purchase:** an item with a sale price above zero, in an order that isn't pending.
- **Free downloader:** a subscriber the sales platform added automatically with the *Free Download* tag.
- **Converted:** their first paid purchase came **more than an hour** after joining the list.

The one-hour rule matters. The sales platform adds buyers to the list at checkout, so most buyers "join" seconds before paying. Those people joined *by* buying, so they don't count as conversions.

## Q1. What share of free downloaders go on to buy?

In [2]:
q1 = run("01_free_download_conversion.sql")
q1

,conversion_pct,one_in
0,1.6,62.0


About **1 in 62** free downloaders later bought a paid licence.

## Q2. How long does it take?

In [3]:
q2 = run("02_time_to_convert.sql")
q2["pct"] = round_to_total(q2["pct"].tolist(), 5)
median_days = run("02b_median_days.sql").iloc[0, 0]
print("Median days from free download to first purchase:", median_days)
q2

Median days from free download to first purchase: 37.0


,time_to_convert,pct
0,1. Within a week,30
1,2. 1 to 4 weeks,15
2,3. 1 to 3 months,20
3,4. 3 to 12 months,35


Only about a quarter bought within a week. **Over half took more than a month**, and about a third took three months or longer. A single welcome email would miss most of them.

## Q3. Do buyers on the list come back more often?

In [4]:
q3 = run("03_repeat_rate_by_path.sql")
ratio = q3.set_index("list_status")["repeat_pct"]
print(f"On the list / not on the list: {ratio['On the list'] / ratio['Not on the list']:.1f}x")
q3

On the list / not on the list: 5.2x


,list_status,repeat_pct
0,On the list,11.5
1,Not on the list,2.2


Buyers on the list are about **five times** as likely to buy again. This is a correlation, not proof the emails cause it: people who unsubscribe drop out of the list, and they are probably less engaged buyers to begin with.

## Q4. Where do new buyers come from?

In [5]:
q4 = run("04_list_share_by_period.sql")
q4["joined_list_first_pct"] = (q4["joined_list_first_pct"] / 5).round() * 5
q4

,period,joined_list_first_pct
0,1. 2021 to 2023,5.0
1,2. 2024 to 2026,20.0


In the peak years, about **1 in 20** new buyers had joined the list before buying. Since 2024, as sales slowed, it's about **1 in 5**. Years are grouped because single recent years are small, and the 2025 sync gap (see Limits) means the recent figure is, if anything, an underestimate.

## Are the differences real?

Q3 and Q4 compare two groups, so before claiming either difference I test whether it could be chance. I use **Fisher's exact test**, which works well for small groups and rare events like these. The cell builds each 2 × 2 table internally and prints only the p-value band, so no group sizes are shown.

In [6]:
from scipy.stats import fisher_exact

def p_band(p):
    for cut in (0.001, 0.01, 0.05):
        if p < cut:
            return f"p < {cut}"
    return f"p = {p:.2f}"

def two_by_two(sql):
    t = pd.read_sql(sql, con).set_index("grp")
    return [[int(t.loc[1, "yes"]), int(t.loc[1, "n"] - t.loc[1, "yes"])],
            [int(t.loc[0, "yes"]), int(t.loc[0, "n"] - t.loc[0, "yes"])]]

repeat_table = two_by_two('''
    SELECT path <> 'not_on_list' AS grp, SUM(paid_orders > 1) AS yes, COUNT(*) AS n
    FROM buyer_paths
    WHERE first_paid_time >= '2021-05-05' AND path <> 'file_import'
    GROUP BY 1''')
period_table = two_by_two('''
    SELECT first_paid_time >= '2024-01-01' AS grp, SUM(path = 'joined_list_first') AS yes, COUNT(*) AS n
    FROM buyer_paths
    WHERE first_paid_time >= '2021-05-05'
    GROUP BY 1''')

tests = {
    "repeat": p_band(fisher_exact(repeat_table).pvalue),
    "list_share": p_band(fisher_exact(period_table).pvalue),
}
print("Q3, repeat buying on vs off the list:", tests["repeat"])
print("Q4, list share 2024 to 2026 vs 2021 to 2023:", tests["list_share"])

Q3, repeat buying on vs off the list: p < 0.001
Q4, list share 2024 to 2026 vs 2021 to 2023: p < 0.001


Both differences are very unlikely to be chance. They are still correlations: significance says the gap is real, not what causes it.

## Q5. Spend per buyer, as an index

In [7]:
q5 = run("05_spend_index_by_path.sql")
q5

,path,spend_index
0,not_on_list,100.0
1,joined_list_first,88.0
2,joined_at_checkout,67.0


Buyers who joined the list first spend about as much as buyers who never subscribed. People added at checkout spend less on average. The data can't show why, so I don't draw a conclusion from it.

## Supporting findings

In [8]:
q6 = run("06_channel_mix_by_year.sql")
q7 = run("07_price_tiers.sql")
q7["pct_of_items"] = round_to_total(q7["pct_of_items"].tolist(), 1)
q8 = run("08_catalogue_concentration.sql")
q9 = run("09_collaboration.sql")

for name, df in [("Channel mix", q6), ("Price bands (USD list price)", q7),
                 ("Catalogue", q8), ("Collaboration", q9)]:
    print(f"\n{name}")
    print(df.to_string(index=False))


Channel mix
      period  own_storefront_pct  marketplace_pct
        2020                42.0             58.0
        2021                65.0             34.0
        2022                53.0             47.0
        2023                39.0             61.0
2024 to 2026                20.0             80.0

Price bands (USD list price)
     price_band  pct_of_items
    1. Under 35            57
    2. 35 to 59            26
    3. 60 to 99             9
4. 100 and over             8

Catalogue
 top_track_pct  top_10_pct  sold_once_pct
          12.0        36.0           68.0

Collaboration
 co_produced_pct
            68.0


## Limits

- The email list export holds **current subscribers only**. Buyers who unsubscribed look as if they were never on the list.
- **A gap in 2025.** Almost nobody joined the list during 2025, against steady sign-ups in every other year, and sign-ups resume in 2026. The most likely cause is that the automatic sync between the sales platform and the email platform stopped for most of that year. Downloaders and buyers from that period are missing from the list, which makes the recent "joined the list first" share lower than it really is.
- Contacts **imported from a spreadsheet in 2021** have their upload date as their join date, so they are excluded from the timing and repeat questions.
- Every finding is a **correlation**. A random holdout test is the way to find causes.

## Save results for the site

In [9]:
results = {
    "conversion_pct": float(q1.loc[0, "conversion_pct"]),
    "one_in": int(q1.loc[0, "one_in"]),
    "median_days": int(median_days),
    "time_to_convert": q2.to_dict("records"),
    "repeat": q3.to_dict("records"),
    "list_share_by_period": q4.to_dict("records"),
    "channel_mix": q6.to_dict("records"),
    "price_bands": q7.to_dict("records"),
    "catalogue": q8.iloc[0].to_dict(),
    "co_produced_pct": float(q9.loc[0, "co_produced_pct"]),
    "tests": tests,
}

out = ROOT / "docs" / "results.json"
out.parent.mkdir(exist_ok=True)
out.write_text(json.dumps(results, indent=2, default=lambda x: x.item() if hasattr(x, "item") else str(x)))
print("Saved", out)

Saved ../docs/results.json
